<sub>**The Annotated ML Handbook** · Part I: Python · by Mahir Absar Khan</sub>

# Chapter 7: The Pythonic toolkit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-1-python/07-python-toolkit.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

Python's standard library ships with tools that turn ten-line loops into one clear line. These are the ones that appear again and again in data and ML code.

## 7.1 collections: smarter containers · *Intermediate*

**Counter, defaultdict, namedtuple, deque**

In [2]:
from collections import Counter, defaultdict, namedtuple, deque

labels = ["cat", "dog", "cat", "bird", "cat", "dog"]
counts = Counter(labels)
print(counts, counts.most_common(1))
print(counts["fish"])

by_group = defaultdict(list)
for name, group in [("ana", "A"), ("bo", "B"), ("cy", "A")]:
    by_group[group].append(name)
print(dict(by_group))

Point = namedtuple("Point", ["x", "y"])
p = Point(2, 5)
print(p.x, p[1], p)

window = deque(maxlen=3)
for loss in [0.9, 0.7, 0.6, 0.55, 0.54]:
    window.append(loss)
print(list(window), sum(window) / len(window))

Counter({'cat': 3, 'dog': 2, 'bird': 1}) [('cat', 3)]
0
{'A': ['ana', 'cy'], 'B': ['bo']}
2 5 Point(x=2, y=5)
[0.6, 0.55, 0.54] 0.5633333333333334


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from collections import Counter, defaultdict, namedtuple, deque</code></td><td>Import <code>Counter</code>, <code>defaultdict</code>, <code>namedtuple</code> and <code>deque</code> from <code>collections</code>.</td></tr>
<tr><td><code>labels = [&quot;cat&quot;, &quot;dog&quot;, &quot;cat&quot;, &quot;bird&quot;, &quot;cat&quot;, &quot;dog&quot;]</code></td><td>Labels to count.</td></tr>
<tr><td><code>counts = Counter(labels)</code></td><td>Counts occurrences of each item.</td></tr>
<tr><td><code>print(counts, counts.most_common(1))</code></td><td><code>most_common(k)</code> returns the top-k (item, count) pairs.</td></tr>
<tr><td><code>print(counts[&quot;fish&quot;])</code></td><td>Missing keys count as 0 instead of raising KeyError.</td></tr>
<tr><td><code>by_group = defaultdict(list)</code></td><td>A dict that creates a default value (here an empty list) for missing keys.</td></tr>
<tr><td><code>for name, group in [(&quot;ana&quot;, &quot;A&quot;), (&quot;bo&quot;, &quot;B&quot;), (&quot;cy&quot;, &quot;A&quot;)]:</code></td><td>Loop over <code>[(&quot;ana&quot;, &quot;A&quot;), (&quot;bo&quot;, &quot;B&quot;), (&quot;cy&quot;, &quot;A&quot;)]</code>, naming each item <code>name, group</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;by_group[group].append(name)</code></td><td>No need to check whether the key exists first.</td></tr>
<tr><td><code>print(dict(by_group))</code></td><td>Convert to a normal dict for printing: {&#x27;A&#x27;: [&#x27;ana&#x27;, &#x27;cy&#x27;], &#x27;B&#x27;: [&#x27;bo&#x27;]}.</td></tr>
<tr><td><code>Point = namedtuple(&quot;Point&quot;, [&quot;x&quot;, &quot;y&quot;])</code></td><td>A tuple whose fields also have names.</td></tr>
<tr><td><code>p = Point(2, 5)</code></td><td>Create a named tuple.</td></tr>
<tr><td><code>print(p.x, p[1], p)</code></td><td>Access by name or by position.</td></tr>
<tr><td><code>window = deque(maxlen=3)</code></td><td>A double-ended queue; with maxlen it keeps only the newest items.</td></tr>
<tr><td><code>for loss in [0.9, 0.7, 0.6, 0.55, 0.54]:</code></td><td>Loop over <code>[0.9, 0.7, 0.6, 0.55, 0.54]</code>, naming each item <code>loss</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;window.append(loss)</code></td><td>When full, appending pushes out the oldest value.</td></tr>
<tr><td><code>print(list(window), sum(window) / len(window))</code></td><td>Moving average of the last 3 losses.</td></tr>
</table>
</details>

## 7.2 itertools: loops without the loops · *Intermediate*

**Grids, combinations and chunks**

In [3]:
from itertools import product, combinations, permutations, chain, islice, accumulate

grid = {"lr": [0.1, 0.01], "depth": [3, 5]}
for lr, depth in product(grid["lr"], grid["depth"]):
    print(f"lr={lr}, depth={depth}")
print(list(combinations(["A", "B", "C"], 2)))
print(len(list(permutations(range(4)))))
print(list(chain([1, 2], (3, 4), {5})))
print(list(islice(range(100), 5, 10)))
print(list(accumulate([1, 2, 3, 4])))

lr=0.1, depth=3
lr=0.1, depth=5
lr=0.01, depth=3
lr=0.01, depth=5
[('A', 'B'), ('A', 'C'), ('B', 'C')]
24
[1, 2, 3, 4, 5]
[5, 6, 7, 8, 9]
[1, 3, 6, 10]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from itertools import product, combinations, permutations, chain, islice, accumulate</code></td><td>Import <code>product</code>, <code>combinations</code>, <code>permutations</code>, <code>chain</code>, <code>islice</code> and <code>accumulate</code> from <code>itertools</code>.</td></tr>
<tr><td><code>grid = {&quot;lr&quot;: [0.1, 0.01], &quot;depth&quot;: [3, 5]}</code></td><td>Candidate values for two hyperparameters.</td></tr>
<tr><td><code>for lr, depth in product(grid[&quot;lr&quot;], grid[&quot;depth&quot;]):</code></td><td>Cartesian product: every combination, a hyperparameter grid.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(f&quot;lr={lr}, depth={depth}&quot;)</code></td><td>Four combinations in total.</td></tr>
<tr><td><code>print(list(combinations([&quot;A&quot;, &quot;B&quot;, &quot;C&quot;], 2)))</code></td><td>All unordered pairs, e.g. pairwise model comparisons.</td></tr>
<tr><td><code>print(len(list(permutations(range(4)))))</code></td><td>Ordered arrangements: 4! = 24.</td></tr>
<tr><td><code>print(list(chain([1, 2], (3, 4), {5})))</code></td><td>Concatenate several iterables lazily.</td></tr>
<tr><td><code>print(list(islice(range(100), 5, 10)))</code></td><td>Slice any iterator (works on generators too).</td></tr>
<tr><td><code>print(list(accumulate([1, 2, 3, 4])))</code></td><td>Running totals: [1, 3, 6, 10].</td></tr>
</table>
</details>

## 7.3 functools: partial and reduce · *Advanced*

**Pre-filling arguments**

In [4]:
from functools import partial, reduce

def scale(x, factor, offset=0):
    return x * factor + offset

to_percent = partial(scale, factor=100)
print(to_percent(0.257), to_percent(0.5, offset=1))
print(reduce(lambda a, b: a * b, [1, 2, 3, 4, 5]))

25.7 51.0
120


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from functools import partial, reduce</code></td><td>Import <code>partial</code> and <code>reduce</code> from <code>functools</code>.</td></tr>
<tr><td><code>def scale(x, factor, offset=0):</code></td><td>Define a function <code>scale</code> that takes <code>x</code>, <code>factor</code> and <code>offset</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return x * factor + offset</code></td><td>Send <code>x * factor + offset</code> back to the caller.</td></tr>
<tr><td><code>to_percent = partial(scale, factor=100)</code></td><td>A new function with <code>factor</code> fixed at 100.</td></tr>
<tr><td><code>print(to_percent(0.257), to_percent(0.5, offset=1))</code></td><td>25.7 and 51.0 (up to float rounding).</td></tr>
<tr><td><code>print(reduce(lambda a, b: a * b, [1, 2, 3, 4, 5]))</code></td><td>Fold a list into one value: ((((1*2)*3)*4)*5) = 120.</td></tr>
</table>
</details>

## 7.4 Context managers · *Intermediate*

> **📖 Definitions**
>
> **Context manager**: an object used with `with` that sets something up on entry and guarantees clean-up on exit, such as closing a file.


Anything used with `with` is a context manager: it sets something up and guarantees clean-up. You can write your own with `contextlib`.

**A timer you can wrap around any code**

In [5]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.4f} s")

with timer("sum of squares"):
    total = sum(i * i for i in range(300_000))

sum of squares: 0.0094 s


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import time</code></td><td>Import the standard-library <code>time</code> module.</td></tr>
<tr><td><code>from contextlib import contextmanager</code></td><td>Import <code>contextmanager</code> from <code>contextlib</code>.</td></tr>
<tr><td><code>@contextmanager</code></td><td>Turn the generator function below into a context manager.</td></tr>
<tr><td><code>def timer(label):</code></td><td>Define a function <code>timer</code> that takes <code>label</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;start = time.perf_counter()</code></td><td>Code before <code>yield</code> runs when the <code>with</code> block starts.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;try:</code></td><td>Start a block whose errors we want to catch.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;yield</code></td><td>The body of the <code>with</code> block runs here.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;finally:</code></td><td>This block always runs, whether or not an error happened.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(f&quot;{label}: {time.perf_counter() - start:.4f} s&quot;)</code></td><td>Runs when the block ends, even after an error.</td></tr>
<tr><td><code>with timer(&quot;sum of squares&quot;):</code></td><td>Everything indented below is timed.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;total = sum(i * i for i in range(300_000))</code></td><td>Some work to time.</td></tr>
</table>
</details>

## 7.5 Randomness and reproducibility · *Beginner*

> **📖 Definitions**
>
> **Random seed**: a starting number for a pseudo-random generator. The same seed always produces the same "random" sequence, which makes results reproducible.
>
> **Reproducibility**: the ability to obtain the same results by re-running the same code on the same data.


ML is full of randomness: data shuffling, train/test splits, weight initialisation. **Seeding** makes it repeatable.

**Seeded random numbers**

In [6]:
import random

random.seed(42)
print(random.random(), random.randint(1, 6))
items = list(range(10))
random.shuffle(items)
print(items, random.choice(items), random.sample(items, 3))
random.seed(42)
print(random.random())

0.6394267984578837 1
[7, 6, 2, 9, 0, 5, 1, 8, 3, 4] 7 [7, 6, 9]
0.6394267984578837


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import random</code></td><td>Import Python&#x27;s built-in <code>random</code> module.</td></tr>
<tr><td><code>random.seed(42)</code></td><td>Fix the starting state of Python&#x27;s random generator.</td></tr>
<tr><td><code>print(random.random(), random.randint(1, 6))</code></td><td>Float in [0, 1) and an integer from 1 to 6 inclusive.</td></tr>
<tr><td><code>items = list(range(10))</code></td><td>The numbers 0..9 as a list.</td></tr>
<tr><td><code>random.shuffle(items)</code></td><td>Shuffle a list in place.</td></tr>
<tr><td><code>print(items, random.choice(items), random.sample(items, 3))</code></td><td>One random item; 3 distinct random items.</td></tr>
<tr><td><code>random.seed(42)</code></td><td>Reset to the same seed.</td></tr>
<tr><td><code>print(random.random())</code></td><td>Same seed -&gt; the very same &quot;random&quot; sequence.</td></tr>
</table>
</details>

> **🔬 Research practice** · **Seed everything, and report it**
>
> For a reproducible experiment you seed Python's `random`, NumPy's generator, and your framework (`random_state=` in scikit-learn, `tf.keras.utils.set_random_seed()` in TensorFlow). Then report results over several seeds (mean ± standard deviation), not one lucky run. [Chapter 29](../part-7-ml-for-research/29-research-protocol.ipynb) shows how.

## 7.6 Dates and times · *Beginner*

**datetime basics**

In [7]:
from datetime import datetime, timedelta, date

d = datetime(2026, 3, 14, 9, 30)
print(d.isoformat(), d.strftime("%d %b %Y"))
print(d + timedelta(days=30))
parsed = datetime.strptime("2026-01-05", "%Y-%m-%d")
print((d - parsed).days, date(2026, 3, 14).weekday())

2026-03-14T09:30:00 14 Mar 2026
2026-04-13 09:30:00
68 5


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from datetime import datetime, timedelta, date</code></td><td>Import <code>datetime</code>, <code>timedelta</code> and <code>date</code> from <code>datetime</code>.</td></tr>
<tr><td><code>d = datetime(2026, 3, 14, 9, 30)</code></td><td>Year, month, day, hour, minute.</td></tr>
<tr><td><code>print(d.isoformat(), d.strftime(&quot;%d %b %Y&quot;))</code></td><td>ISO 8601 string; custom format (%d day, %b short month, %Y year).</td></tr>
<tr><td><code>print(d + timedelta(days=30))</code></td><td>Date arithmetic with timedelta.</td></tr>
<tr><td><code>parsed = datetime.strptime(&quot;2026-01-05&quot;, &quot;%Y-%m-%d&quot;)</code></td><td>Parse a string with a known format.</td></tr>
<tr><td><code>print((d - parsed).days, date(2026, 3, 14).weekday())</code></td><td>Difference in days; weekday: Monday = 0.</td></tr>
</table>
</details>

## 7.7 Regular expressions · *Intermediate*

> **📖 Definitions**
>
> **Regular expression (regex)**: a pattern language for finding and extracting text, such as `\d+` for "one or more digits".


Regular expressions find patterns in text: IDs, dates, numbers buried in strings. They are also how pandas' `.str.extract()` works.

**Extracting structured values from messy text**

In [8]:
import re

text = "Sample S-104 measured 12.5 mg on 2026-02-01; sample S-221 measured 7 mg."
ids = re.findall(r"S-\d+", text)
doses = re.findall(r"(\d+(?:\.\d+)?) mg", text)
date_m = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
print(ids, [float(x) for x in doses], date_m.groups())
print(re.sub(r"\s+", " ", "too    many   spaces"))

['S-104', 'S-221'] [12.5, 7.0] ('2026', '02', '01')
too many spaces


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import re</code></td><td>Import the regular-expression module <code>re</code>.</td></tr>
<tr><td><code>text = &quot;Sample S-104 measured 12.5 mg on 2026-02-01; sample S-221 measured 7 mg.&quot;</code></td><td>Messy text with IDs, doses and a date.</td></tr>
<tr><td><code>ids = re.findall(r&quot;S-\d+&quot;, text)</code></td><td><code>\d+</code> = one or more digits. <code>r&quot;...&quot;</code> raw strings keep backslashes literal.</td></tr>
<tr><td><code>doses = re.findall(r&quot;(\d+(?:\.\d+)?) mg&quot;, text)</code></td><td>A group (...) captures just the number; (?:...) groups without capturing.</td></tr>
<tr><td><code>date_m = re.search(r&quot;(\d{4})-(\d{2})-(\d{2})&quot;, text)</code></td><td><code>search</code> returns the first match or None.</td></tr>
<tr><td><code>print(ids, [float(x) for x in doses], date_m.groups())</code></td><td>IDs, doses converted to floats, and the captured date parts.</td></tr>
<tr><td><code>print(re.sub(r&quot;\s+&quot;, &quot; &quot;, &quot;too    many   spaces&quot;))</code></td><td>Replace runs of whitespace with a single space.</td></tr>
</table>
</details>

## 7.8 Measuring speed · *Intermediate*

**Timing small snippets**

In [9]:
import timeit

loop_time = timeit.timeit("s = 0\nfor i in range(1000): s += i", number=2000)
builtin_time = timeit.timeit("sum(range(1000))", number=2000)
print(f"loop {loop_time:.3f}s vs sum() {builtin_time:.3f}s -> {loop_time / builtin_time:.1f}x faster")

loop 0.075s vs sum() 0.033s -> 2.2x faster


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import timeit</code></td><td>Import the <code>timeit</code> module.</td></tr>
<tr><td><code>loop_time = timeit.timeit(&quot;s = 0\nfor i in range(1000): s += i&quot;, number=2000)</code></td><td>Run the snippet 2000 times; total seconds.</td></tr>
<tr><td><code>builtin_time = timeit.timeit(&quot;sum(range(1000))&quot;, number=2000)</code></td><td>The same total using the built-in <code>sum</code>.</td></tr>
<tr><td><code>print(f&quot;loop {loop_time:.3f}s vs sum() {builtin_time:.3f}s -&gt; {loop_time / builtin_time:.1f}x faster&quot;)</code></td><td>Compare the two timings.</td></tr>
</table>
</details>

In Jupyter, `%timeit expression` does the same in one line, and `%%time` times a whole cell. The next part shows how NumPy makes numeric code tens to hundreds of times faster than pure Python loops.

> **✍️ Try it yourself**
>
> 1. Use `Counter` to find the three most common words in a paragraph (lowercase it and split on whitespace first).
> 2. Use `itertools.product` to list all 12 combinations of `C in [0.1, 1, 10]` and `kernel in ["linear", "rbf", "poly", "sigmoid"]`.
> 3. Write a regex that extracts all e-mail addresses from a string.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 7*.

---

⬅️ [Chapter 6: Errors, files and modules](06-python-errors-files.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 8: NumPy arrays: shape, dtype and indexing](../part-2-numpy/08-numpy-arrays.ipynb) ➡️